# Orchestrator Agent â€” Qwen3-8B INT4 on Kaggle T4

**Orchestrator's job â€” final decision maker:**
1. Read Red Team report (what attack family, how many samples)
2. Read Blue Team report (evasion scores, severity, retrain list)
3. Read evasion report (raw detector scores)
4. Decide routing: retrain / partial_retrain / fast_promote
5. Set env vars (AI_ACTION, ARGO_WORKFLOW, SEVERITY)
6. Trigger Argo workflow
7. Write memory (update attack_memory.json for next round)

**Orchestrator has the most tools (8) â€” context budget matters most here.**

Threshold decisions Qwen3 makes (same as Haiku production rules):
- `evasion > 0.40` â†’ CRITICAL â†’ `full-canary`
- `0.25-0.40` â†’ HIGH â†’ `full-canary`
- `< 0.25` â†’ LOW â†’ `fast-promote`
- Multiple detectors `> 0.40` â†’ `emergency-rollback`

**Qwen3 `temperature=0.1`** â€” most deterministic setting. Orchestrator decisions affect production
deployments via Argo â€” we want repeatable, grounded-in-numbers decisions.

**Run on:** Kaggle T4 GPU | **Secrets:** `HF_TOKEN`, `ARGO_TOKEN` (optional)

---

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q", "-U",
    "langchain-huggingface>=0.1.2", "bitsandbytes>=0.46.1",
    "langchain>=0.2.0", "langgraph>=0.1.0",
    "huggingface_hub>=0.23.0", "accelerate>=0.27.0",
], check=True)
print("Install complete.")

## Load Qwen3-8B INT4

`temperature=0.1` â€” Orchestrator needs fully deterministic tool calls.
Routing decisions affect real Argo workflows. Wrong JSON = wrong deployment.

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

MODEL_ID = "Qwen/Qwen3-8B"
KAGGLE_CACHE = Path("/kaggle/input/qwen3-8b-cache")
MODEL_SOURCE = str(KAGGLE_CACHE) if KAGGLE_CACHE.exists() else MODEL_ID

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE â€” need T4'}")
print(f"Model source: {'Kaggle cache (fast ~2min)' if KAGGLE_CACHE.exists() else 'HuggingFace download (~15min)'}")

qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)
qwen_model = AutoModelForCausalLM.from_pretrained(MODEL_SOURCE, quantization_config=bnb, device_map="auto")

vram = torch.cuda.memory_allocated() / 1e9
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {vram:.1f} / {total_vram:.1f} GB ({vram/total_vram:.0%})")

hf_pipe = pipeline(
    "text-generation", model=qwen_model, tokenizer=qwen_tokenizer,
    max_new_tokens=512, temperature=0.1, do_sample=True, return_full_text=False,
)
llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=hf_pipe))
print("LLM ready. temperature=0.1 (max determinism for routing decisions).")

In [ ]:
import os, sys, subprocess
from pathlib import Path

ROUND = int(os.environ.get("ROUND", "1"))
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
ARGO_TOKEN = os.environ.get("ARGO_TOKEN", "")
REPO_DIR = Path("/kaggle/working/repo")

if not REPO_DIR.exists():
    print("Cloning repo from GitHub...")
    subprocess.run([
        "git", "clone",
        "https://github.com/av4874/Orchestration.git",
        str(REPO_DIR),
    ], check=True, capture_output=True)
    print(f"Cloned to {REPO_DIR}")

os.environ["ENTERPRISE_ROOT"] = str(REPO_DIR)
os.environ["ADVERSARIAL_DATASET"] = "Builder117/enterprise-adversarial-samples"
os.environ["HF_TOKEN"] = HF_TOKEN
if ARGO_TOKEN:
    os.environ["ARGO_TOKEN"] = ARGO_TOKEN
sys.path.insert(0, str(REPO_DIR))
for d in ["results", "agent_workspace", "agent_traces"]:
    (REPO_DIR / d).mkdir(exist_ok=True)
print(f"ENTERPRISE_ROOT={REPO_DIR} | Round={ROUND} | Argo: {'LIVE' if ARGO_TOKEN else 'dry_run'}")

## Tool truncation

Orchestrator has 8 tools â€” the most of any agent.
`read_evasion_report` and `read_attack_memory` can return large JSON.
Without truncation, reading both could consume ~3k tokens in 2 steps,
leaving only ~18k tokens for the remaining 6 tool calls.

With 400-char truncation:
- 8 tools Ã— 100 tokens each = 800 tokens for tool results
- 8 AI reasoning steps Ã— 100 tokens = 800 tokens
- Total: ~1800 tokens â€” 5.6% of 32k budget

In [ ]:
import time
from langchain.tools import StructuredTool
from langchain.callbacks.base import BaseCallbackHandler
from agents.tools.analysis_tools import read_evasion_report, analyze_weakness
from agents.tools.comms_tools import read_message
from agents.tools.memory_tools import read_attack_memory, write_memory
from agents.tools.routing_tools import decide_routing, set_env_vars, trigger_argo

MAX_RESULT_CHARS = 400

def truncate_tool(t):
    def fn(**kwargs):
        r = t.invoke(kwargs)
        if isinstance(r, str) and len(r) > MAX_RESULT_CHARS:
            return r[:MAX_RESULT_CHARS] + "...[truncated]"
        return r
    return StructuredTool.from_function(fn, name=t.name, description=t.description, args_schema=t.args_schema)

RAW_TOOLS = [read_attack_memory, read_evasion_report, analyze_weakness,
             read_message, decide_routing, set_env_vars, trigger_argo, write_memory]
TOOLS = [truncate_tool(t) for t in RAW_TOOLS]

class StepTracker(BaseCallbackHandler):
    def __init__(self, tok):
        self.steps = []; self.t0 = time.time(); self._tok = tok; self._cum = 0
    def _n(self, t): return len(self._tok.encode(str(t)))
    def on_llm_end(self, response, **kw):
        text = ""
        try: text = response.generations[0][0].text or ""
        except: pass
        n = self._n(text); self._cum += n
        self.steps.append({"type":"AI","label":"Qwen3","tokens":n,"cumulative":self._cum,
                           "content":text[:300],"elapsed":round(time.time()-self.t0,1)})
        print(f"  [AI #{len(self.steps)}] {n} tokens | {text[:100].strip()}")
    def on_tool_end(self, output, **kw):
        n = self._n(output); self._cum += n
        name = kw.get("name", "tool")
        self.steps.append({"type":"Tool","label":name,"tokens":n,"cumulative":self._cum,
                           "content":str(output)[:300],"elapsed":round(time.time()-self.t0,1)})
        print(f"  [Tool:{name}] {n} tokens | {str(output)[:80]}")

tracker = StepTracker(qwen_tokenizer)
print(f"Tools ({len(TOOLS)}): {[t.name for t in TOOLS]}")

## Run Orchestrator ReAct loop

This is the same `create_react_agent` call as production.
Qwen3 reads both agent reports, synthesizes evasion scores, and makes the routing decision.
It then sets env vars and triggers Argo â€” the same steps Jenkins would verify in `pipeline_decision.json`.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

SYSTEM_PROMPT = """You are the Pipeline Orchestrator. Read agent reports, decide routing, trigger Argo.

Thresholds: evasion >0.40=CRITICAL/full-canary | 0.25-0.40=HIGH/full-canary | <0.25=LOW/fast-promote | multiple >0.40=emergency-rollback.

TOOLS (in order):
1. read_message from red_team
2. read_message from blue_team
3. read_evasion_report
4. analyze_weakness {\"detector\":\"all\"} â€” once only
5. decide_routing {action,severity,argo_workflow,confidence,reason}
6. set_env_vars {AI_ACTION,ARGO_WORKFLOW,SEVERITY}
7. trigger_argo {workflow,round,dry_run}
8. write_memory with round summary

Ground decisions in numbers. No repeated tool calls."""

agent = create_react_agent(llm, TOOLS, prompt=SYSTEM_PROMPT)

print("=" * 60)
print(f"ORCHESTRATOR AGENT â€” ROUND {ROUND} â€” Qwen3-8B INT4")
print("=" * 60)

result = agent.invoke(
    {"messages": [HumanMessage(content=(
        f"Execute orchestration for round {ROUND}. "
        "Call tools only. "
        "Step 1: read_message from red_team. "
        "Step 2: read_message from blue_team. "
        "Step 3: read_evasion_report. "
        "Step 4: decide_routing. "
        "Step 5: set_env_vars. "
        f"Step 6: trigger_argo dry_run={'true' if not ARGO_TOKEN else 'false'}. "
        "Step 7: write_memory."
    ))]},
    config={"recursion_limit": 10, "callbacks": [tracker]},
)

print("\n" + "=" * 60)
print(f"Final: {result['messages'][-1].content[:400]}")

## Visualize â€” Token growth, tool timeline, decision trace

Orchestrator calls 8 tools â€” the most complex ReAct trace in the pipeline.
The charts show exactly what Qwen3 decided and in what order.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

steps = tracker.steps
if steps:
    labels = [f"{i}: {s['label'][:10]}" for i, s in enumerate(steps)]
    colors = ["steelblue" if s["type"]=="AI" else "coral" for s in steps]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Tokens per step
    axes[0].bar(range(len(steps)), [s["tokens"] for s in steps], color=colors)
    axes[0].set_xticks(range(len(steps)))
    axes[0].set_xticklabels(labels, rotation=35, ha="right", fontsize=7)
    axes[0].set_title("Tokens Per Step"); axes[0].set_ylabel("tokens")
    axes[0].legend(handles=[mpatches.Patch(color="steelblue",label="AI reasoning"),
                             mpatches.Patch(color="coral",label="Tool result")])
    axes[0].grid(axis="y", alpha=0.3)

    # 2. Context growth vs 32k limit
    cum = [s["cumulative"] for s in steps]
    axes[1].fill_between(range(len(steps)), cum, alpha=0.3, color="purple")
    axes[1].plot(range(len(steps)), cum, marker="o", color="purple", linewidth=2)
    axes[1].axhline(32000, color="red", linestyle="--", linewidth=1.5, label="32k limit (Qwen3)")
    axes[1].axhline(200000, color="orange", linestyle=":", linewidth=1, label="200k limit (Haiku)")
    axes[1].set_title("Context Window Growth"); axes[1].set_ylabel("cumulative tokens")
    axes[1].set_xticks(range(len(steps)))
    axes[1].set_xticklabels(labels, rotation=35, ha="right", fontsize=7)
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

    # 3. Tool-only timeline (horizontal bars)
    tool_steps = [s for s in steps if s["type"]=="Tool"]
    if tool_steps:
        tool_names = list(dict.fromkeys(s["label"] for s in tool_steps))
        cmap = plt.cm.get_cmap("tab10", len(tool_names))
        nc = {n: cmap(i) for i, n in enumerate(tool_names)}
        for i, s in enumerate(tool_steps):
            axes[2].barh(i, 1, left=s["elapsed"], height=0.6, color=nc[s["label"]], edgecolor="black", lw=0.5)
            axes[2].text(s["elapsed"]+0.05, i, s["label"], va="center", fontsize=7)
        axes[2].set_xlabel("Elapsed seconds"); axes[2].set_ylabel("Tool call #")
        axes[2].set_title("Tool Call Timeline")
        axes[2].set_yticks(range(len(tool_steps)))
        axes[2].set_yticklabels([f"#{i+1}" for i in range(len(tool_steps))])
        patches = [mpatches.Patch(color=nc[n], label=n) for n in tool_names]
        axes[2].legend(handles=patches, fontsize=7, loc="lower right")

    plt.suptitle(f"Orchestrator Agent â€” Round {ROUND} â€” Qwen3-8B INT4", fontsize=13)
    plt.tight_layout()
    plt.savefig("/kaggle/working/orchestrator_viz.png", dpi=120)
    plt.show()

    total = cum[-1]
    print(f"\nTotal tokens: {total:,} / 32,000 ({total/32000:.1%} of Qwen3 limit)")
    print(f"Tool calls made: {len(tool_steps)} / 8 tools available")

# Decision trace
print("\n" + "=" * 50 + "\nORCHESTRATOR DECISION TRACE\n" + "=" * 50)
for i, s in enumerate(steps):
    tag = "REASONING" if s["type"]=="AI" else f"TOOL:{s['label']}"
    print(f"\nStep {i+1} [{tag}] +{s['tokens']} tokens @ {s['elapsed']}s")
    print(s["content"][:350])
    print("-" * 40)

In [ ]:
import json
from datetime import datetime, timezone
from huggingface_hub import upload_file
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")

trace = {
    "round": ROUND, "mode": "live_qwen3", "agent": "orchestrator",
    "model": "Qwen/Qwen3-8B-INT4",
    "total_tokens": tracker.steps[-1]["cumulative"] if tracker.steps else 0,
    "steps": len(tracker.steps),
    "final_output": result["messages"][-1].content[:500],
    "timestamp": datetime.now(timezone.utc).isoformat(),
}
trace_path = REPO_DIR / f"agent_traces/round_{ROUND}_orchestrator.json"
trace_path.write_text(json.dumps(trace, indent=2), encoding="utf-8")

files_to_push = [
    (trace_path, f"agent_traces/round_{ROUND}_orchestrator.json"),
    (REPO_DIR / "results/pipeline_decision.json", "results/pipeline_decision.json"),
    (REPO_DIR / "pipeline/attack_memory.json", "pipeline/attack_memory.json"),
]

for local, remote in files_to_push:
    if Path(local).exists():
        upload_file(path_or_fileobj=str(local), path_in_repo=remote,
                    repo_id="Builder117/enterprise-adversarial-samples",
                    repo_type="dataset", token=HF_TOKEN)
        print(f"Pushed: {remote}")

print(f"\nOrchestrator done | Round {ROUND} | {trace['total_tokens']:,} tokens used")
print(f"Context: {trace['total_tokens']/32000:.1%} of Qwen3 32k limit")